# Expérience 2

Cette expérience complète `experience_1.ipynb` avec une stratégie **dédiée aux séries temporelles**.
L'objectif n'est pas de réutiliser le dataset large en colonnes, mais de reconstruire un **panel annuel long** par couple `area + crop`, afin d'évaluer des modèles temporels locaux comparables dans MLflow.

Modèles comparés :

- `naive_last_value` : baseline temporelle simple ;
- `holt_linear` : lissage exponentiel avec tendance ;
- `sarimax_univariate` : SARIMAX sans variables exogènes ;
- `sarimax_exog` : SARIMAX avec pluie, pesticides et température comme variables exogènes.


## Principes de comparabilité avec l'expérience 1

On conserve volontairement la même logique de suivi que dans `experience_1.ipynb` :

- un notebook autonome ;
- un dossier d'artefacts dédié sous `artifacts/experiments/experience_2/` ;
- un run MLflow par modèle candidat ;
- un run de synthèse final ;
- les mêmes métriques principales : `train_mae`, `train_rmse`, `train_r2`, `test_mae`, `test_rmse`, `test_r2`, `overfit_gap_rmse`, `overfit_ratio_rmse`.

La différence de fond est le **protocole d'évaluation** :

- expérience 1 : généralisation sur des pays absents du train avec `GroupShuffleSplit(area)` ;
- expérience 2 : généralisation **dans le temps** avec un holdout sur les dernières années de chaque série.


In [1]:
from __future__ import annotations

from datetime import date
from pathlib import Path
import json
import shutil
import sqlite3
import warnings

import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

from scripts.project_config import DEFAULT_CONFIG_PATH, load_preparation_config

pd.set_option('display.max_columns', None)
SEED = 42
warnings.filterwarnings('ignore', category=UserWarning, module='statsmodels')
warnings.filterwarnings('ignore', category=ConvergenceWarning)


## Configuration

Le notebook relit la configuration centralisée du projet, prépare un dossier d'artefacts dédié et pointe vers la même base MLflow locale que le reste du projet.


In [2]:
config = load_preparation_config(ensure_dirs=True)
MIN_YEAR = int(config['MIN_YEAR'])
CURRENT_YEAR = date.today().year

YIELD_PATH = config['YIELD_PATH']
RAINFALL_PATH = config['RAINFALL_PATH']
PESTICIDES_PATH = config['PESTICIDES_PATH']
TEMP_PATH = config['TEMP_PATH']
ARTIFACTS_DIR = config['ARTIFACTS_DIR']

EXOG_COLUMNS = ['average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp']
TEST_HORIZON = 4
MIN_TRAIN_POINTS = 12
MIN_SERIES_LENGTH = MIN_TRAIN_POINTS + TEST_HORIZON

EXPERIENCE_DIR = ARTIFACTS_DIR / 'experiments' / 'experience_2'
EXPERIENCE_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR = EXPERIENCE_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

EXPERIENCE_DATASET_PATH = EXPERIENCE_DIR / 'dataset_series_temporelles.csv'
SOURCE_OVERVIEW_PATH = EXPERIENCE_DIR / 'source_overview.csv'
SOURCE_QUALITY_PATH = EXPERIENCE_DIR / 'source_quality.csv'
SERIES_SUMMARY_PATH = EXPERIENCE_DIR / 'series_summary.csv'
SPLIT_OVERVIEW_PATH = EXPERIENCE_DIR / 'split_overview.csv'
EXPERIENCE_SUMMARY_PATH = EXPERIENCE_DIR / 'experience_2_summary.csv'
MISSING_SUMMARY_PATH = EXPERIENCE_DIR / 'experience_2_missing_summary.csv'
MODEL_RESULTS_PATH = EXPERIENCE_DIR / 'model_results.csv'

MLFLOW_DB_PATH = ARTIFACTS_DIR / 'mlflow.db'
MLFLOW_TRACKING_URI = f"sqlite:///{MLFLOW_DB_PATH.resolve()}"
MLFLOW_EXPERIMENT_NAME = 'experience_2'
MLFLOW_ARTIFACTS_DIR = ARTIFACTS_DIR / 'mlruns'
MLFLOW_EXPERIMENT_ARTIFACT_DIR = MLFLOW_ARTIFACTS_DIR / MLFLOW_EXPERIMENT_NAME
MLFLOW_EXPERIMENT_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Configuration chargée depuis : {DEFAULT_CONFIG_PATH.resolve()}')
print(f'Fenêtre historique globale : {MIN_YEAR}-{CURRENT_YEAR}')
print(f'Dossier expérience : {EXPERIENCE_DIR.resolve()}')
print(f'Tracking MLflow : {MLFLOW_TRACKING_URI}')


Configuration chargée depuis : /Users/steph/Code/Python/Jupyter/OCR_Projet12/config/project_paths.yaml
Fenêtre historique globale : 1990-2026
Dossier expérience : /Users/steph/Code/Python/Jupyter/OCR_Projet12/artifacts/experiments/experience_2
Tracking MLflow : sqlite:////Users/steph/Code/Python/Jupyter/OCR_Projet12/artifacts/mlflow.db


## Chargement et harmonisation des sources annuelles

Comme dans l'expérience 1, on repart des quatre tables annuelles historiques.
La différence est qu'on garde ici une structure **longue** par année, indispensable pour entraîner des modèles temporels locaux.


In [3]:
yield_source = pd.read_csv(YIELD_PATH)
rainfall_source = pd.read_csv(RAINFALL_PATH, na_values=['..'])
pesticides_source = pd.read_csv(PESTICIDES_PATH)
temp_source = pd.read_csv(TEMP_PATH)

source_overview = pd.DataFrame(
    [
        {'fichier': 'yield.csv', 'lignes': yield_source.shape[0], 'colonnes': yield_source.shape[1], 'nan_detectes': int(yield_source.isna().sum().sum())},
        {'fichier': 'rainfall.csv', 'lignes': rainfall_source.shape[0], 'colonnes': rainfall_source.shape[1], 'nan_detectes': int(rainfall_source.isna().sum().sum())},
        {'fichier': 'pesticides.csv', 'lignes': pesticides_source.shape[0], 'colonnes': pesticides_source.shape[1], 'nan_detectes': int(pesticides_source.isna().sum().sum())},
        {'fichier': 'temp.csv', 'lignes': temp_source.shape[0], 'colonnes': temp_source.shape[1], 'nan_detectes': int(temp_source.isna().sum().sum())},
    ]
)
source_overview.to_csv(SOURCE_OVERVIEW_PATH, index=False)

yield_clean = (
    yield_source.loc[:, ['Area', 'Item', 'Year', 'Value']]
    .rename(columns={'Area': 'area', 'Item': 'crop', 'Year': 'year', 'Value': 'target_yield_t_ha'})
    .assign(
        area=lambda df: df['area'].astype('string').str.strip(),
        crop=lambda df: df['crop'].astype('string').str.strip(),
        year=lambda df: pd.to_numeric(df['year'], errors='coerce').astype('Int64'),
        target_yield_t_ha=lambda df: pd.to_numeric(df['target_yield_t_ha'], errors='coerce') / 10000,
    )
    .dropna(subset=['area', 'crop', 'year'])
)
yield_clean = yield_clean.loc[yield_clean['year'].between(MIN_YEAR, CURRENT_YEAR, inclusive='both')].copy()
yield_clean['year'] = yield_clean['year'].astype(int)

rainfall_clean = (
    rainfall_source.loc[:, [' Area', 'Year', 'average_rain_fall_mm_per_year']]
    .rename(columns={' Area': 'area', 'Year': 'year'})
    .assign(
        area=lambda df: df['area'].astype('string').str.strip(),
        year=lambda df: pd.to_numeric(df['year'], errors='coerce').astype('Int64'),
        average_rain_fall_mm_per_year=lambda df: pd.to_numeric(df['average_rain_fall_mm_per_year'], errors='coerce'),
    )
    .dropna(subset=['area', 'year'])
)
rainfall_clean = rainfall_clean.loc[rainfall_clean['year'].between(MIN_YEAR, CURRENT_YEAR, inclusive='both')].copy()
rainfall_clean['year'] = rainfall_clean['year'].astype(int)
rainfall_clean = rainfall_clean.drop_duplicates(subset=['area', 'year'], keep='first')

pesticides_clean = (
    pesticides_source.loc[:, ['Area', 'Year', 'Value']]
    .rename(columns={'Area': 'area', 'Year': 'year', 'Value': 'pesticides_tonnes'})
    .assign(
        area=lambda df: df['area'].astype('string').str.strip(),
        year=lambda df: pd.to_numeric(df['year'], errors='coerce').astype('Int64'),
        pesticides_tonnes=lambda df: pd.to_numeric(df['pesticides_tonnes'], errors='coerce'),
    )
    .dropna(subset=['area', 'year'])
)
pesticides_clean = pesticides_clean.loc[pesticides_clean['year'].between(MIN_YEAR, CURRENT_YEAR, inclusive='both')].copy()
pesticides_clean['year'] = pesticides_clean['year'].astype(int)
pesticides_clean = pesticides_clean.drop_duplicates(subset=['area', 'year'], keep='first')

temp_clean = (
    temp_source.loc[:, ['year', 'country', 'avg_temp']]
    .rename(columns={'country': 'area'})
    .assign(
        area=lambda df: df['area'].astype('string').str.strip(),
        year=lambda df: pd.to_numeric(df['year'], errors='coerce').astype('Int64'),
        avg_temp=lambda df: pd.to_numeric(df['avg_temp'], errors='coerce'),
    )
    .dropna(subset=['area', 'year'])
)
temp_clean = temp_clean.loc[temp_clean['year'].between(MIN_YEAR, CURRENT_YEAR, inclusive='both')].copy()
temp_clean['year'] = temp_clean['year'].astype(int)
temp_clean = temp_clean.groupby(['area', 'year'], as_index=False)['avg_temp'].mean()

source_quality = pd.DataFrame(
    [
        {
            'source': 'yield',
            'rows_after_cleaning': len(yield_clean),
            'min_year': int(yield_clean['year'].min()),
            'max_year': int(yield_clean['year'].max()),
            'duplicate_key_count': int(yield_clean.duplicated(['area', 'crop', 'year']).sum()),
        },
        {
            'source': 'rainfall',
            'rows_after_cleaning': len(rainfall_clean),
            'min_year': int(rainfall_clean['year'].min()),
            'max_year': int(rainfall_clean['year'].max()),
            'duplicate_key_count': int(rainfall_clean.duplicated(['area', 'year']).sum()),
        },
        {
            'source': 'pesticides',
            'rows_after_cleaning': len(pesticides_clean),
            'min_year': int(pesticides_clean['year'].min()),
            'max_year': int(pesticides_clean['year'].max()),
            'duplicate_key_count': int(pesticides_clean.duplicated(['area', 'year']).sum()),
        },
        {
            'source': 'temp',
            'rows_after_cleaning': len(temp_clean),
            'min_year': int(temp_clean['year'].min()),
            'max_year': int(temp_clean['year'].max()),
            'duplicate_key_count': int(temp_clean.duplicated(['area', 'year']).sum()),
        },
    ]
)
source_quality.to_csv(SOURCE_QUALITY_PATH, index=False)

display(source_overview)
display(source_quality)


,fichier,lignes,colonnes,nan_detectes
0,yield.csv,56717,12,0
1,rainfall.csv,6727,3,780
2,pesticides.csv,4349,7,0
3,temp.csv,71311,3,2547


,source,rows_after_cleaning,min_year,max_year,duplicate_key_count
0,yield,29367,1990,2016,0
1,rainfall,5859,1990,2017,0
2,pesticides,4349,1990,2016,0
3,temp,3288,1990,2013,0


## Construction du panel temporel

Le dataset final contient une ligne par triplet `area + crop + year`.
Les variables exogènes sont imputées par interpolation / `ffill` / `bfill` **au niveau du pays**, ce qui évite d'introduire une fuite d'information entre cultures d'un même pays.

Pour garder une comparaison stricte entre modèles, on retient ensuite un **même sous-ensemble de séries éligibles** :

- longueur minimale suffisante pour entraîner puis tester ;
- série annuelle continue ;
- dernière année alignée sur l'année maximale disponible ;
- couverture complète des variables exogènes après imputation.


In [4]:
exogenous_panel = (
    rainfall_clean
    .merge(pesticides_clean, on=['area', 'year'], how='outer', validate='1:1')
    .merge(temp_clean, on=['area', 'year'], how='outer', validate='1:1')
    .sort_values(['area', 'year'])
    .reset_index(drop=True)
)

for column in EXOG_COLUMNS:
    exogenous_panel[column] = (
        exogenous_panel.groupby('area')[column]
        .transform(lambda s: s.interpolate(limit_direction='both').ffill().bfill())
    )

experience_2_dataset = (
    yield_clean
    .merge(exogenous_panel, on=['area', 'year'], how='left', validate='m:1')
    .sort_values(['area', 'crop', 'year'])
    .reset_index(drop=True)
)

MAX_AVAILABLE_YEAR = int(experience_2_dataset['year'].max())
HOLDOUT_START_YEAR = MAX_AVAILABLE_YEAR - TEST_HORIZON + 1

missing_summary = (
    experience_2_dataset.isna()
    .sum()
    .rename('nb_nan')
    .reset_index()
    .rename(columns={'index': 'variable'})
)
missing_summary['part_nan_pct'] = (missing_summary['nb_nan'] / len(experience_2_dataset) * 100).round(2)
missing_summary.to_csv(MISSING_SUMMARY_PATH, index=False)

series_summary = (
    experience_2_dataset.groupby(['area', 'crop'])
    .agg(
        n_obs=('year', 'size'),
        start_year=('year', 'min'),
        end_year=('year', 'max'),
        unique_years=('year', 'nunique'),
        target_non_null=('target_yield_t_ha', lambda s: int(s.notna().sum())),
    )
    .reset_index()
)
series_summary['is_consecutive'] = series_summary['unique_years'] == (series_summary['end_year'] - series_summary['start_year'] + 1)
series_summary['ends_on_last_year'] = series_summary['end_year'] == MAX_AVAILABLE_YEAR
series_summary['exog_missing_after_imputation'] = (
    experience_2_dataset.groupby(['area', 'crop'])[EXOG_COLUMNS]
    .apply(lambda df: int(df.isna().sum().sum()))
    .to_numpy()
)
series_summary['eligible_for_experience_2'] = (
    (series_summary['n_obs'] >= MIN_SERIES_LENGTH)
    & series_summary['is_consecutive']
    & series_summary['ends_on_last_year']
    & (series_summary['target_non_null'] == series_summary['n_obs'])
    & (series_summary['exog_missing_after_imputation'] == 0)
)
series_summary.to_csv(SERIES_SUMMARY_PATH, index=False)

eligible_keys = series_summary.loc[series_summary['eligible_for_experience_2'], ['area', 'crop']].copy()
eligible_panel_df = (
    experience_2_dataset
    .merge(eligible_keys, on=['area', 'crop'], how='inner', validate='m:1')
    .sort_values(['area', 'crop', 'year'])
    .reset_index(drop=True)
)

experience_summary = pd.DataFrame(
    {
        'indicateur': [
            'nb_lignes_panel',
            'nb_colonnes_panel',
            'max_available_year',
            'holdout_start_year',
            'test_horizon',
            'min_train_points',
            'min_series_length',
            'nb_series_total',
            'nb_series_eligibles',
            'nb_areas',
            'nb_crops',
            'part_nan_globale_pct',
        ],
        'valeur': [
            int(experience_2_dataset.shape[0]),
            int(experience_2_dataset.shape[1]),
            MAX_AVAILABLE_YEAR,
            HOLDOUT_START_YEAR,
            TEST_HORIZON,
            MIN_TRAIN_POINTS,
            MIN_SERIES_LENGTH,
            int(series_summary.shape[0]),
            int(series_summary['eligible_for_experience_2'].sum()),
            int(experience_2_dataset['area'].nunique()),
            int(experience_2_dataset['crop'].nunique()),
            round(experience_2_dataset.isna().mean().mean() * 100, 2),
        ],
    }
)
experience_summary.to_csv(EXPERIENCE_SUMMARY_PATH, index=False)
experience_2_dataset.to_csv(EXPERIENCE_DATASET_PATH, index=False)

display(experience_summary)
display(series_summary.head(10))
print(f'Dataset panel sauvegardé : {EXPERIENCE_DATASET_PATH.resolve()}')


,indicateur,valeur
0,nb_lignes_panel,29367.00
1,nb_colonnes_panel,7.00
2,max_available_year,2016.00
3,holdout_start_year,2013.00
4,test_horizon,4.00
5,min_train_points,12.00
6,min_series_length,16.00
7,nb_series_total,1168.00
8,nb_series_eligibles,567.00
9,nb_areas,212.00


,area,crop,n_obs,start_year,end_year,unique_years,target_non_null,is_consecutive,ends_on_last_year,exog_missing_after_imputation,eligible_for_experience_2
0,Afghanistan,Maize,27,1990,2016,27,27,True,True,27,False
1,Afghanistan,Potatoes,27,1990,2016,27,27,True,True,27,False
2,Afghanistan,"Rice, paddy",27,1990,2016,27,27,True,True,27,False
3,Afghanistan,Wheat,27,1990,2016,27,27,True,True,27,False
4,Albania,Maize,27,1990,2016,27,27,True,True,0,True
5,Albania,Potatoes,27,1990,2016,27,27,True,True,0,True
6,Albania,"Rice, paddy",4,1990,1993,4,4,True,False,0,False
7,Albania,Sorghum,3,1990,1992,3,3,True,False,0,False
8,Albania,Soybeans,27,1990,2016,27,27,True,True,0,True
9,Albania,Wheat,27,1990,2016,27,27,True,True,0,True


Dataset panel sauvegardé : /Users/steph/Code/Python/Jupyter/OCR_Projet12/artifacts/experiments/experience_2/dataset_series_temporelles.csv


## Protocole temporel retenu

Chaque série `area + crop` est évaluée avec le même schéma :

- entraînement sur toutes les années avant le holdout ;
- test sur les `4` dernières années disponibles ;
- agrégation des prédictions de toutes les séries pour obtenir les métriques globales.

Ce protocole rend les modèles comparables entre eux et cohérents avec une vraie logique de prévision.


In [5]:
split_overview = pd.DataFrame(
    {
        'indicateur': [
            'train_min_year',
            'train_max_year',
            'test_min_year',
            'test_max_year',
            'eligible_series_count',
            'eligible_panel_rows',
        ],
        'valeur': [
            int(eligible_panel_df['year'].min()),
            int(HOLDOUT_START_YEAR - 1),
            int(HOLDOUT_START_YEAR),
            int(MAX_AVAILABLE_YEAR),
            int(series_summary['eligible_for_experience_2'].sum()),
            int(eligible_panel_df.shape[0]),
        ],
    }
)
split_overview.to_csv(SPLIT_OVERVIEW_PATH, index=False)

display(split_overview)
display(eligible_panel_df.head(10))


,indicateur,valeur
0,train_min_year,1990
1,train_max_year,2012
2,test_min_year,2013
3,test_max_year,2016
4,eligible_series_count,567
5,eligible_panel_rows,15126


,area,crop,year,target_yield_t_ha,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,Albania,Maize,1990,3.6613,1485.0,121.00,16.37
1,Albania,Maize,1991,2.9068,1485.0,121.00,15.36
2,Albania,Maize,1992,2.4876,1485.0,121.00,16.06
3,Albania,Maize,1993,2.4185,1485.0,121.00,16.05
4,Albania,Maize,1994,2.5848,1485.0,201.00,16.96
5,Albania,Maize,1995,3.1300,1485.0,251.00,15.67
6,Albania,Maize,1996,3.2604,1485.0,313.96,15.64
7,Albania,Maize,1997,3.1862,1485.0,376.93,15.90
8,Albania,Maize,1998,3.3416,1485.0,439.89,16.27
9,Albania,Maize,1999,3.7455,1485.0,502.86,16.57


## Modèles temporels locaux

Chaque candidat est entraîné **série par série**.
C'est un choix important : on respecte la structure d'une vraie série temporelle locale, au lieu de forcer un modèle tabulaire global sur un panel annualisé.

Remarque : `sarimax_exog` standardise les variables exogènes **par série** avant estimation pour améliorer la stabilité numérique.


In [6]:
def split_single_series(series_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    ordered = series_df.sort_values('year').reset_index(drop=True)
    train_df = ordered.iloc[:-TEST_HORIZON].copy()
    test_df = ordered.iloc[-TEST_HORIZON:].copy()
    return train_df, test_df

def make_prediction_frame(source_df: pd.DataFrame, predictions, *, split: str) -> pd.DataFrame:
    pred_values = np.asarray(predictions, dtype=float).reshape(-1)
    frame = source_df[['area', 'crop', 'year', 'target_yield_t_ha']].copy().reset_index(drop=True)
    if len(pred_values) < len(frame):
        frame = frame.iloc[-len(pred_values):].copy().reset_index(drop=True)
    elif len(pred_values) > len(frame):
        pred_values = pred_values[-len(frame):]
    frame['prediction'] = pred_values
    frame = frame.rename(columns={'target_yield_t_ha': 'actual'})
    frame['split'] = split
    if split == 'test':
        frame['forecast_horizon'] = np.arange(1, len(frame) + 1)
    else:
        frame['forecast_horizon'] = 0
    frame = frame.replace([np.inf, -np.inf], np.nan)
    return frame.dropna(subset=['actual', 'prediction']).reset_index(drop=True)

def scale_exog(train_df: pd.DataFrame, test_df: pd.DataFrame):
    train_exog = train_df[EXOG_COLUMNS].astype(float).copy()
    test_exog = test_df[EXOG_COLUMNS].astype(float).copy()
    means = train_exog.mean()
    stds = train_exog.std(ddof=0).replace(0.0, 1.0).fillna(1.0)
    return (train_exog - means) / stds, (test_exog - means) / stds

def fit_naive_last_value(train_df: pd.DataFrame, test_df: pd.DataFrame) -> dict[str, object]:
    train_series = train_df['target_yield_t_ha'].astype(float).reset_index(drop=True)
    train_predictions = train_series.shift(1).iloc[1:]
    train_frame = make_prediction_frame(train_df.iloc[1:].copy(), train_predictions.to_numpy(), split='train')
    last_observed_value = float(train_series.iloc[-1])
    test_predictions = np.repeat(last_observed_value, len(test_df))
    test_frame = make_prediction_frame(test_df, test_predictions, split='test')
    return {
        'train_predictions': train_frame,
        'test_predictions': test_frame,
        'fit_metadata': {'strategy': 'last_train_observation'},
    }

def fit_holt_linear(train_df: pd.DataFrame, test_df: pd.DataFrame) -> dict[str, object]:
    train_series = train_df['target_yield_t_ha'].astype(float).reset_index(drop=True)
    model = ExponentialSmoothing(
        train_series,
        trend='add',
        damped_trend=True,
        seasonal=None,
        initialization_method='estimated',
    )
    fitted = model.fit(optimized=True, use_brute=False)
    train_frame = make_prediction_frame(train_df, fitted.fittedvalues, split='train')
    test_frame = make_prediction_frame(test_df, fitted.forecast(len(test_df)), split='test')
    return {
        'train_predictions': train_frame,
        'test_predictions': test_frame,
        'fit_metadata': {'trend': 'add', 'damped_trend': True},
    }

def fit_sarimax_univariate(train_df: pd.DataFrame, test_df: pd.DataFrame) -> dict[str, object]:
    train_series = train_df['target_yield_t_ha'].astype(float).reset_index(drop=True)
    model = SARIMAX(
        endog=train_series,
        order=(1, 0, 0),
        trend='c',
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fitted = model.fit(disp=False, maxiter=200)
    train_frame = make_prediction_frame(train_df, fitted.fittedvalues, split='train')
    test_forecast = fitted.get_forecast(steps=len(test_df)).predicted_mean
    test_frame = make_prediction_frame(test_df, test_forecast, split='test')
    return {
        'train_predictions': train_frame,
        'test_predictions': test_frame,
        'fit_metadata': {'order': [1, 0, 0], 'trend': 'c', 'uses_exog': False},
    }

def fit_sarimax_exog(train_df: pd.DataFrame, test_df: pd.DataFrame) -> dict[str, object]:
    train_series = train_df['target_yield_t_ha'].astype(float).reset_index(drop=True)
    train_exog, test_exog = scale_exog(train_df, test_df)
    model = SARIMAX(
        endog=train_series,
        exog=train_exog,
        order=(1, 0, 0),
        trend='c',
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fitted = model.fit(disp=False, maxiter=200)
    train_frame = make_prediction_frame(train_df, fitted.fittedvalues, split='train')
    test_forecast = fitted.get_forecast(steps=len(test_df), exog=test_exog).predicted_mean
    test_frame = make_prediction_frame(test_df, test_forecast, split='test')
    return {
        'train_predictions': train_frame,
        'test_predictions': test_frame,
        'fit_metadata': {'order': [1, 0, 0], 'trend': 'c', 'uses_exog': True, 'exog_columns': EXOG_COLUMNS},
    }

def compute_regression_metrics(predictions_df: pd.DataFrame) -> dict[str, float]:
    if predictions_df.empty:
        return {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan}
    actual = predictions_df['actual'].to_numpy(dtype=float)
    predicted = predictions_df['prediction'].to_numpy(dtype=float)
    return {
        'mae': float(mean_absolute_error(actual, predicted)),
        'rmse': float(np.sqrt(mean_squared_error(actual, predicted))),
        'r2': float(r2_score(actual, predicted)) if len(predictions_df) >= 2 else np.nan,
    }

def compute_horizon_metrics(test_predictions_df: pd.DataFrame) -> pd.DataFrame:
    if test_predictions_df.empty:
        return pd.DataFrame(columns=['forecast_horizon', 'mae', 'rmse', 'r2', 'n_predictions'])
    rows = []
    for horizon, horizon_df in test_predictions_df.groupby('forecast_horizon', sort=True):
        metrics = compute_regression_metrics(horizon_df)
        rows.append(
            {
                'forecast_horizon': int(horizon),
                'n_predictions': int(len(horizon_df)),
                'mae': metrics['mae'],
                'rmse': metrics['rmse'],
                'r2': metrics['r2'],
            }
        )
    return pd.DataFrame(rows)

def evaluate_local_model(model_name: str, model_spec: dict[str, object], panel_df: pd.DataFrame) -> dict[str, object]:
    train_predictions = []
    test_predictions = []
    failures = []
    series_logs = []

    for (area, crop), series_df in panel_df.groupby(['area', 'crop'], sort=True):
        ordered_series = series_df.sort_values('year').reset_index(drop=True)
        train_df, test_df = split_single_series(ordered_series)
        try:
            result = model_spec['fit_function'](train_df, test_df)
            train_predictions.append(result['train_predictions'])
            test_predictions.append(result['test_predictions'])
            series_logs.append(
                {
                    'area': area,
                    'crop': crop,
                    'train_rows': int(len(train_df)),
                    'test_rows': int(len(test_df)),
                    'status': 'ok',
                }
            )
        except Exception as exc:
            failures.append(
                {
                    'area': area,
                    'crop': crop,
                    'error_type': type(exc).__name__,
                    'error_message': str(exc)[:500],
                }
            )
            series_logs.append(
                {
                    'area': area,
                    'crop': crop,
                    'train_rows': int(len(train_df)),
                    'test_rows': int(len(test_df)),
                    'status': 'failed',
                }
            )

    train_predictions_df = pd.concat(train_predictions, ignore_index=True) if train_predictions else pd.DataFrame(columns=['area', 'crop', 'year', 'actual', 'prediction', 'split', 'forecast_horizon'])
    test_predictions_df = pd.concat(test_predictions, ignore_index=True) if test_predictions else pd.DataFrame(columns=['area', 'crop', 'year', 'actual', 'prediction', 'split', 'forecast_horizon'])
    all_predictions_df = pd.concat([train_predictions_df, test_predictions_df], ignore_index=True)
    failures_df = pd.DataFrame(failures)
    series_logs_df = pd.DataFrame(series_logs)
    horizon_metrics_df = compute_horizon_metrics(test_predictions_df)

    train_metrics = compute_regression_metrics(train_predictions_df)
    test_metrics = compute_regression_metrics(test_predictions_df)
    overfit_gap_rmse = np.nan
    overfit_ratio_rmse = np.nan
    if pd.notna(train_metrics['rmse']) and pd.notna(test_metrics['rmse']):
        overfit_gap_rmse = float(test_metrics['rmse'] - train_metrics['rmse'])
        if train_metrics['rmse'] > 0:
            overfit_ratio_rmse = float(test_metrics['rmse'] / train_metrics['rmse'])

    results_row = {
        'model': model_name,
        'model_family': model_spec['model_family'],
        'uses_exog': bool(model_spec['uses_exog']),
        'eligible_series': int(panel_df.groupby(['area', 'crop']).ngroups),
        'successful_series': int(series_logs_df['status'].eq('ok').sum()),
        'failed_series': int(series_logs_df['status'].eq('failed').sum()),
        'train_mae': train_metrics['mae'],
        'train_rmse': train_metrics['rmse'],
        'train_r2': train_metrics['r2'],
        'test_mae': test_metrics['mae'],
        'test_rmse': test_metrics['rmse'],
        'test_r2': test_metrics['r2'],
        'overfit_gap_rmse': overfit_gap_rmse,
        'overfit_ratio_rmse': overfit_ratio_rmse,
        'train_predictions_count': int(len(train_predictions_df)),
        'test_predictions_count': int(len(test_predictions_df)),
    }

    return {
        'results_row': results_row,
        'all_predictions_df': all_predictions_df,
        'train_predictions_df': train_predictions_df,
        'test_predictions_df': test_predictions_df,
        'failures_df': failures_df,
        'series_logs_df': series_logs_df,
        'horizon_metrics_df': horizon_metrics_df,
    }


## Entraînement et suivi MLflow

Le mécanisme de tracking reprend la structure de l'expérience 1.
Chaque modèle journalise ses métriques et ses artefacts, notamment les prédictions détaillées et les éventuels échecs série par série.


In [7]:
candidate_models = {
    'naive_last_value': {
        'model_family': 'naive',
        'uses_exog': False,
        'fit_function': fit_naive_last_value,
        'specification': {'strategy': 'last_train_observation'},
    },
    'holt_linear': {
        'model_family': 'exponential_smoothing',
        'uses_exog': False,
        'fit_function': fit_holt_linear,
        'specification': {'trend': 'add', 'damped_trend': True},
    },
    'sarimax_univariate': {
        'model_family': 'sarimax',
        'uses_exog': False,
        'fit_function': fit_sarimax_univariate,
        'specification': {'order': [1, 0, 0], 'trend': 'c', 'uses_exog': False},
    },
    'sarimax_exog': {
        'model_family': 'sarimax',
        'uses_exog': True,
        'fit_function': fit_sarimax_exog,
        'specification': {'order': [1, 0, 0], 'trend': 'c', 'uses_exog': True, 'exog_columns': EXOG_COLUMNS},
    },
}

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
while mlflow.active_run() is not None:
    mlflow.end_run()

tracking_db_path = Path(MLFLOW_TRACKING_URI.removeprefix('sqlite:///'))
experiment_artifact_uri = MLFLOW_EXPERIMENT_ARTIFACT_DIR.resolve().as_uri()
tracking_db_path.parent.mkdir(parents=True, exist_ok=True)
MLFLOW_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

if tracking_db_path.exists():
    connection = sqlite3.connect(tracking_db_path)
    cursor = connection.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='experiments'")
    if cursor.fetchone() is not None:
        cursor.execute(
            'SELECT experiment_id, name, artifact_location FROM experiments WHERE name = ?',
            (MLFLOW_EXPERIMENT_NAME,),
        )
        existing_row = cursor.fetchone()
        if existing_row is not None:
            experiment_id, _, current_artifact_location = existing_row
            current_artifact_dir = Path(str(current_artifact_location).removeprefix('file://')).resolve()
            target_artifact_dir = MLFLOW_EXPERIMENT_ARTIFACT_DIR.resolve()
            if current_artifact_dir.exists() and current_artifact_dir != target_artifact_dir:
                for child in current_artifact_dir.iterdir():
                    destination = target_artifact_dir / child.name
                    if not destination.exists():
                        shutil.move(str(child), str(destination))
                if current_artifact_dir.exists() and current_artifact_dir.is_dir() and not any(current_artifact_dir.iterdir()):
                    current_artifact_dir.rmdir()
            cursor.execute(
                'UPDATE experiments SET artifact_location = ? WHERE experiment_id = ?',
                (experiment_artifact_uri, experiment_id),
            )
            cursor.execute(
                'UPDATE runs SET artifact_uri = REPLACE(artifact_uri, ?, ?) WHERE experiment_id = ? AND artifact_uri LIKE ?',
                (
                    str(current_artifact_dir),
                    str(target_artifact_dir),
                    experiment_id,
                    f'{current_artifact_dir}%',
                ),
            )
            connection.commit()
    connection.close()

client = mlflow.tracking.MlflowClient()
if client.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME) is None:
    client.create_experiment(MLFLOW_EXPERIMENT_NAME, artifact_location=experiment_artifact_uri)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

results_rows = []

for model_name, model_spec in candidate_models.items():
    evaluation = evaluate_local_model(model_name, model_spec, eligible_panel_df)

    predictions_path = PREDICTIONS_DIR / f'{model_name}_predictions.csv'
    failures_path = PREDICTIONS_DIR / f'{model_name}_failures.csv'
    horizon_metrics_path = PREDICTIONS_DIR / f'{model_name}_horizon_metrics.csv'
    series_logs_path = PREDICTIONS_DIR / f'{model_name}_series_logs.csv'
    model_spec_path = PREDICTIONS_DIR / f'{model_name}_specification.json'

    evaluation['all_predictions_df'].to_csv(predictions_path, index=False)
    evaluation['failures_df'].to_csv(failures_path, index=False)
    evaluation['horizon_metrics_df'].to_csv(horizon_metrics_path, index=False)
    evaluation['series_logs_df'].to_csv(series_logs_path, index=False)
    model_spec_path.write_text(
        json.dumps(model_spec['specification'], ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

    with mlflow.start_run(run_name=f'experience_2__{model_name}') as run:
        mlflow.log_param('experience_name', 'experience_2')
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('model_family', model_spec['model_family'])
        mlflow.log_param('split_strategy', 'last_4_years_per_series')
        mlflow.log_param('panel_granularity', 'area_crop_year')
        mlflow.log_param('uses_exog', bool(model_spec['uses_exog']))
        mlflow.log_param('test_horizon', TEST_HORIZON)
        mlflow.log_param('min_train_points', MIN_TRAIN_POINTS)
        mlflow.log_param('min_series_length', MIN_SERIES_LENGTH)
        mlflow.log_param('max_available_year', MAX_AVAILABLE_YEAR)
        mlflow.log_param('eligible_series', evaluation['results_row']['eligible_series'])
        mlflow.log_param('successful_series', evaluation['results_row']['successful_series'])
        mlflow.log_param('failed_series', evaluation['results_row']['failed_series'])
        mlflow.log_param('exog_columns', ','.join(EXOG_COLUMNS) if model_spec['uses_exog'] else 'none')

        mlflow.log_metric('train_mae', evaluation['results_row']['train_mae'])
        mlflow.log_metric('train_rmse', evaluation['results_row']['train_rmse'])
        mlflow.log_metric('train_r2', evaluation['results_row']['train_r2'])
        mlflow.log_metric('test_mae', evaluation['results_row']['test_mae'])
        mlflow.log_metric('test_rmse', evaluation['results_row']['test_rmse'])
        mlflow.log_metric('test_r2', evaluation['results_row']['test_r2'])
        mlflow.log_metric('overfit_gap_rmse', evaluation['results_row']['overfit_gap_rmse'])
        mlflow.log_metric('overfit_ratio_rmse', evaluation['results_row']['overfit_ratio_rmse'])
        mlflow.log_metric('train_predictions_count', evaluation['results_row']['train_predictions_count'])
        mlflow.log_metric('test_predictions_count', evaluation['results_row']['test_predictions_count'])

        mlflow.log_artifact(str(EXPERIENCE_SUMMARY_PATH))
        mlflow.log_artifact(str(MISSING_SUMMARY_PATH))
        mlflow.log_artifact(str(SPLIT_OVERVIEW_PATH))
        mlflow.log_artifact(str(SERIES_SUMMARY_PATH))
        mlflow.log_artifact(str(predictions_path))
        mlflow.log_artifact(str(failures_path))
        mlflow.log_artifact(str(horizon_metrics_path))
        mlflow.log_artifact(str(series_logs_path))
        mlflow.log_artifact(str(model_spec_path))

        results_row = evaluation['results_row'].copy()
        results_row['run_id'] = run.info.run_id
        results_rows.append(results_row)

if not results_rows:
    raise RuntimeError('Aucun modèle temporel n a produit de résultat exploitable.')

results_df = pd.DataFrame(results_rows).sort_values(['test_rmse', 'test_mae'], ascending=[True, True]).reset_index(drop=True)
results_df.to_csv(MODEL_RESULTS_PATH, index=False)

with mlflow.start_run(run_name='experience_2__summary'):
    mlflow.log_param('experience_name', 'experience_2')
    mlflow.log_param('models_tested', ','.join(candidate_models.keys()))
    mlflow.log_param('time_split_strategy', 'last_4_years_per_series')
    mlflow.log_metric('best_test_rmse', float(results_df.loc[0, 'test_rmse']))
    mlflow.log_metric('best_test_r2', float(results_df.loc[0, 'test_r2']))
    mlflow.log_artifact(str(MODEL_RESULTS_PATH))
    mlflow.log_artifact(str(EXPERIENCE_SUMMARY_PATH))
    mlflow.log_artifact(str(SERIES_SUMMARY_PATH))

display(results_df)


,model,model_family,uses_exog,eligible_series,successful_series,failed_series,train_mae,train_rmse,train_r2,test_mae,test_rmse,test_r2,overfit_gap_rmse,overfit_ratio_rmse,train_predictions_count,test_predictions_count,run_id
0,naive_last_value,naive,False,567,567,0,0.659127,1.405142,0.968705,0.949033,2.033192,0.949611,0.628050,1.446965,12291,2268,ed723bb821a7416986bb0c2690d141b2
1,holt_linear,exponential_smoothing,False,567,567,0,0.531372,1.109799,0.980265,0.929009,2.064239,0.948061,0.954439,1.860011,12858,2268,c9a87484ec594e0cb8863f19ea850395
2,sarimax_univariate,sarimax,False,567,567,0,0.833587,2.261123,0.918078,1.122953,2.552597,0.920578,0.291473,1.128906,12858,2268,1febc51562b34caeac27c220afca1535
3,sarimax_exog,sarimax,True,567,567,0,0.814666,2.380214,0.909222,1.243725,2.776057,0.906064,0.395843,1.166306,12858,2268,8f8cbf933d7f4edbb8d5a5896e5055df


## Lecture finale

L'intérêt de cette expérience n'est pas seulement de comparer des métriques.
Elle sert aussi à répondre à une question de fond : **un modèle explicitement temporel apporte-t-il un gain par rapport à l'approche tabulaire de l'expérience 1 ?**

Le meilleur modèle ici sera donc à interpréter en regard des runs MLflow de `experience_1`, pas isolément.


In [8]:
best_model_name = results_df.loc[0, 'model']
best_run_id = results_df.loc[0, 'run_id']
best_horizon_metrics_path = PREDICTIONS_DIR / f'{best_model_name}_horizon_metrics.csv'
best_horizon_metrics_df = pd.read_csv(best_horizon_metrics_path) if best_horizon_metrics_path.exists() else pd.DataFrame()

display(results_df)
if not best_horizon_metrics_df.empty:
    display(best_horizon_metrics_df)

print('Lecture métier proposée :')
print('- comparer d abord le meilleur `test_rmse` de cette expérience avec celui de `experience_1` ;')
print('- vérifier ensuite le nombre de séries en échec pour juger la robustesse opérationnelle ;')
print('- lire enfin la dérive par horizon via `*_horizon_metrics.csv` : si la qualité chute vite, le modèle est peu utile pour une planification pluriannuelle.')
print()
print(f'Meilleur modèle temporel : {best_model_name}')
print(f'Run MLflow du meilleur modèle : {best_run_id}')
print(f'Années de test : {HOLDOUT_START_YEAR}-{MAX_AVAILABLE_YEAR}')


,model,model_family,uses_exog,eligible_series,successful_series,failed_series,train_mae,train_rmse,train_r2,test_mae,test_rmse,test_r2,overfit_gap_rmse,overfit_ratio_rmse,train_predictions_count,test_predictions_count,run_id
0,naive_last_value,naive,False,567,567,0,0.659127,1.405142,0.968705,0.949033,2.033192,0.949611,0.628050,1.446965,12291,2268,ed723bb821a7416986bb0c2690d141b2
1,holt_linear,exponential_smoothing,False,567,567,0,0.531372,1.109799,0.980265,0.929009,2.064239,0.948061,0.954439,1.860011,12858,2268,c9a87484ec594e0cb8863f19ea850395
2,sarimax_univariate,sarimax,False,567,567,0,0.833587,2.261123,0.918078,1.122953,2.552597,0.920578,0.291473,1.128906,12858,2268,1febc51562b34caeac27c220afca1535
3,sarimax_exog,sarimax,True,567,567,0,0.814666,2.380214,0.909222,1.243725,2.776057,0.906064,0.395843,1.166306,12858,2268,8f8cbf933d7f4edbb8d5a5896e5055df


,forecast_horizon,n_predictions,mae,rmse,r2
0,1,567,0.761910,1.688579,0.964879
1,2,567,0.916014,1.839428,0.960617
2,3,567,1.030767,2.222592,0.938417
3,4,567,1.087441,2.315333,0.933681


Lecture métier proposée :
- comparer d abord le meilleur `test_rmse` de cette expérience avec celui de `experience_1` ;
- vérifier ensuite le nombre de séries en échec pour juger la robustesse opérationnelle ;
- lire enfin la dérive par horizon via `*_horizon_metrics.csv` : si la qualité chute vite, le modèle est peu utile pour une planification pluriannuelle.

Meilleur modèle temporel : naive_last_value
Run MLflow du meilleur modèle : ed723bb821a7416986bb0c2690d141b2
Années de test : 2013-2016
